# Predicting votes of individual members of parliament

Things  to do:
* sklearn nlp transformers
* extract `party_is_in_government` feature for members from legislature id
* extract `bill_proposed_by_government` feature from poll description
* depending on performance try encoding of poll description / poll title with language models

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn import (
    compose,
    dummy,
    ensemble,
    metrics,
    pipeline,
    preprocessing,
    set_config,
)

import bundestag.logging as b_logging
import bundestag.modelling as modelling

In [ ]:
set_config(transform_output="pandas")

In [ ]:
aw_path = Path("../data/preprocessed/abgeordnetenwatch")

In [ ]:
all_votes = pd.concat(
    [
        pd.read_parquet(aw_path / "df_all_votes_111.parquet"),
        pd.read_parquet(aw_path / "df_all_votes_132.parquet"),
    ],
    axis=0,
    ignore_index=True,
)
display(all_votes.head(), all_votes.tail())

In [ ]:
# all_votes["politician name"] = all_votes["mandate"].str.extract(
#     r"(.*)\s\(Bundestag", expand=True
# )[0]

In [ ]:
polls = pd.concat(
    [
        pd.read_parquet(aw_path / "df_polls_111.parquet"),
        pd.read_parquet(aw_path / "df_polls_132.parquet"),
    ],
    ignore_index=True,
    axis=0,
)
display(polls.head(2), polls.tail(2))

In [ ]:
polls["poll_date"] = pd.to_datetime(polls["poll_date"], format="%Y-%m-%d")
polls["poll_date"].head()

In [ ]:
polls["poll_description_without_results"] = polls["poll_description"].apply(
    lambda x: "\n".join(x.split("\n")[:-1])
)
polls["poll_description_without_results"].head()

In [ ]:
mandates = pd.concat(
    [
        pd.read_parquet(aw_path / "df_mandates_111.parquet"),
        pd.read_parquet(aw_path / "df_mandates_132.parquet"),
    ],
    ignore_index=True,
    axis=0,
)
display(mandates.head(2), mandates.tail(2))

In [ ]:
tmp = all_votes.merge(
    mandates[["mandate_id", "party", "politician"]],
    on="mandate_id",
    how="left",
)

display(tmp.head(2), tmp.tail(2))

In [ ]:
all_info = all_votes.merge(
    polls[
        [
            "poll_id",
            "poll_date",
            "poll_title",
            "poll_description_without_results",
            "legislature_period",
        ]
    ],
    on="poll_id",
    how="left",
)
all_info = all_info.merge(
    mandates[["mandate_id", "party", "politician"]],
    on="mandate_id",
    how="left",
)

if "politician name" in all_info.columns:
    all_info.drop(columns=["politician name"], inplace=True)

display(all_info.head(2), all_info.tail(2))

In [ ]:
target = "vote"

In [ ]:
mandates.head(2)

Unfortunately there are entries for members of parliament in the votes API which are not present in the mandates API for the same legislative period, e.g. Katharina Barley for Bundestag 2017 - 2021 (legislature_id 111). Because of this we drop all entries in the votes dataset that have no matching entry in the mandates dataset.

In [ ]:
logger = b_logging.logger

In [ ]:
n0 = len(all_info)
all_info = all_info.copy().query("party.notna()")
n1 = len(all_info)
logger.info(
    f"dropped {n0-n1:_d} rows due to missing party, keeping {n1/n0:.2%}"
)

In [ ]:
y = all_info[target].copy()
y.head()

In [ ]:
x_cols = [
    "mandate_id",
    "poll_date",
    "poll_title",
    "poll_description_without_results",
    "legislature_period",
    "party",
    "poll_id",
]
X = all_info[x_cols].copy()
X.head()

## Time holdout split

In [ ]:
all_info["poll_date"].value_counts().sort_index().plot()

In [ ]:
t_split = "2022-01-01"
t_split

In [ ]:
mask_train = X["poll_date"] < t_split
mask_test = ~mask_train

X0 = X.loc[mask_train, :].copy()
X1 = X.loc[mask_test, :].copy()
y0 = y.loc[mask_train].copy()
y1 = y.loc[mask_test].copy()

In [ ]:
y1

## Create a baseline model

In [ ]:
dummy_model = dummy.DummyClassifier(strategy="prior")
dummy_model.fit(X0, y0)

In [ ]:
performance_dummy = metrics.classification_report(
    y1, dummy_model.predict(X1), output_dict=True
)

print(json.dumps(performance_dummy, indent=2))

```json
{
  "abstain": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 1161
  },
  "no": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 16109
  },
  "no_show": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 4791
  },
  "yes": {
    "precision": 0.5267504719409645,
    "recall": 1.0,
    "f1-score": 0.6900282418400754,
    "support": 24555
  },
  "accuracy": 0.5267504719409645,
  "macro avg": {
    "precision": 0.13168761798524112,
    "recall": 0.25,
    "f1-score": 0.17250706046001885,
    "support": 46616
  },
  "weighted avg": {
    "precision": 0.2774660596900288,
    "recall": 0.5267504719409645,
    "f1-score": 0.36347270204185367,
    "support": 46616
  }
}
```

## Hand-crafted features
> `party_is_in_government` and `bill_proposed_by_government` and `party`

### Tree-based

In [ ]:
X0.head(2)

In [ ]:
X0["legislature_period"].nunique()

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "mandate_id",
                    "poll_title",
                    "poll_description_without_results",
                    "legislature_period",
                    "poll_id",
                ]
            ),
        ),
        (
            "encode",
            compose.make_column_transformer(
                (
                    preprocessing.OneHotEncoder(sparse_output=False),
                    [
                        "party",
                    ],
                ),
                remainder="passthrough",
            ),
        ),
    ]
)

X0_features = trafo.fit_transform(X0)
display(X0_features.head(), X0_features.tail())

In [ ]:
pd.concat((X0, X0_features, y0), axis=1)[
    [
        "legislature_period",
        "party",
        "remainder__party_is_in_government",
        "poll_description_without_results",
        "remainder__bill_proposed_by_government",
        "vote",
    ]
].sample(2)

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        ("estimator", ensemble.RandomForestClassifier()),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_handcrafted = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_handcrafted, indent=2))

```json
{
  "abstain": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 1162
  },
  "no": {
    "precision": 0.4192466131566654,
    "recall": 0.6092698882922916,
    "f1-score": 0.4967044025157233,
    "support": 16203
  },
  "no_show": {
    "precision": 0.0,
    "recall": 0.0,
    "f1-score": 0.0,
    "support": 4978
  },
  "yes": {
    "precision": 0.596526244267029,
    "recall": 0.5675096961861668,
    "f1-score": 0.5816563146997928,
    "support": 24752
  },
  "accuracy": 0.5078883108610256,
  "macro avg": {
    "precision": 0.2539432143559236,
    "recall": 0.2941948961196146,
    "f1-score": 0.269590179303879,
    "support": 47095
  },
  "weighted avg": {
    "precision": 0.4577613434775444,
    "recall": 0.5078883108610256,
    "f1-score": 0.4765953611935776,
    "support": 47095
  }
}
```

### FastAI-based

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "poll_title",
                    "poll_description_without_results",
                ]
            ),
        ),
    ]
)

trafo.fit_transform(X0).head()

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        (
            "estimator",
            modelling.FastAIClassifier(
                cat_names_endings=[
                    "mandate_id",
                    "party_is_in_government",
                    "party",
                    "bill_proposed_by_government",
                    "legislature_period",
                ]
            ),
        ),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_handcrafted = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_handcrafted, indent=2))

```json
{
  "abstain": {
    "precision": 0.12924865831842575,
    "recall": 0.24892334194659776,
    "f1-score": 0.1701501324698263,
    "support": 1161
  },
  "no": {
    "precision": 0.37681549220010757,
    "recall": 0.5218201005649016,
    "f1-score": 0.43761876252700627,
    "support": 16109
  },
  "no_show": {
    "precision": 0.20075627363355106,
    "recall": 0.1218952202045502,
    "f1-score": 0.15168831168831168,
    "support": 4791
  },
  "yes": {
    "precision": 0.6336168658352033,
    "recall": 0.49448177560578294,
    "f1-score": 0.5554691431447001,
    "support": 24555
  },
  "accuracy": 0.4595203363651965,
  "macro avg": {
    "precision": 0.3351093224968219,
    "recall": 0.34678010958045813,
    "f1-score": 0.32873158745746106,
    "support": 46616
  },
  "weighted avg": {
    "precision": 0.48782529397033175,
    "recall": 0.4595203363651965,
    "f1-score": 0.4636482851502021,
    "support": 46616
  }
}
```

### Tree-based with TFIDF

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(
                description_col="poll_description_without_results"
            ),
        ),
        (
            "tfidf",
            modelling.DenseTfidfVectorizer(
                text_col="poll_description_without_results",
                poll_id_col="poll_id",
                no_below=10,
                no_above=0.5,
                keep_n=500,
            ),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "mandate_id",
                    "poll_description_without_results",
                    "legislature_period",
                    "poll_id",
                    "poll_title",
                ]
            ),
        ),
        (
            "encode",
            compose.make_column_transformer(
                (
                    preprocessing.OneHotEncoder(sparse_output=False),
                    [
                        "party",
                    ],
                ),
                remainder="passthrough",
            ),
        ),
    ]
)

trafo.fit_transform(X0).head()

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        ("estimator", ensemble.RandomForestClassifier()),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_tfidf_tree = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_tfidf_tree, indent=2))

```json
{
  "abstain": {
    "precision": 0.11254019292604502,
    "recall": 0.030146425495262703,
    "f1-score": 0.04755434782608695,
    "support": 1161
  },
  "no": {
    "precision": 0.45081735620585267,
    "recall": 0.6933391271959775,
    "f1-score": 0.5463751100675082,
    "support": 16109
  },
  "no_show": {
    "precision": 0.3229166666666667,
    "recall": 0.0064704654560634525,
    "f1-score": 0.012686719869040311,
    "support": 4791
  },
  "yes": {
    "precision": 0.6681440701688905,
    "recall": 0.5832213398493179,
    "f1-score": 0.6228011046119724,
    "support": 24555
  },
  "accuracy": 0.5482237858246096,
  "macro avg": {
    "precision": 0.3886045714918637,
    "recall": 0.3282943394991554,
    "f1-score": 0.307354320593652,
    "support": 46616
  },
  "weighted avg": {
    "precision": 0.543724200877474,
    "recall": 0.5482237858246096,
    "f1-score": 0.5193588133804515,
    "support": 46616
  }
}
```

### FastAI-based with TFIDF

In [ ]:
trafo = pipeline.Pipeline(
    [
        ("party_is_in_government", modelling.PartyInGovernmentTransformer()),
        (
            "bill_proposed_by_government",
            modelling.BillProposedByGovernmentTransformer(
                description_col="poll_description_without_results"
            ),
        ),
        (
            "tfidf",
            modelling.DenseTfidfVectorizer(
                text_col="poll_description_without_results",
                poll_id_col="poll_id",
                no_below=10,
                no_above=0.5,
                keep_n=500,
            ),
        ),
        (
            "filter",
            modelling.ColumnRemoverTransformer(
                columns_to_remove=[
                    "poll_date",
                    "poll_description_without_results",
                    "poll_title",
                ],
                column_beginnings=[],
            ),
        ),  # "tfidf_poll_title"
    ]
)

X0_trafo = trafo.fit_transform(X0)
X0_trafo.head(2)

In [ ]:
model_handcrafted = pipeline.Pipeline(
    [
        ("transformer", trafo),
        (
            "estimator",
            modelling.FastAIClassifier(
                cat_names_endings=[
                    "mandate_id",
                    "party_is_in_government",
                    "party",
                    "bill_proposed_by_government",
                    "legislature_period",
                ],
                cont_names_beginnings=["tfidf_poll_description_"],
                batch_size=2048,
                n_epochs=5,
            ),
        ),
    ]
)

In [ ]:
model_handcrafted.fit(X0, y0)

In [ ]:
performance_tfidf_fastai = metrics.classification_report(
    y1, model_handcrafted.predict(X1), output_dict=True
)
print(json.dumps(performance_tfidf_fastai, indent=2))

```json
{
  "abstain": {
    "precision": 0.01839339339339339,
    "recall": 0.04220499569336779,
    "f1-score": 0.02562091503267974,
    "support": 1161
  },
  "no": {
    "precision": 0.5120219613782658,
    "recall": 0.6715500651809547,
    "f1-score": 0.5810349920777721,
    "support": 16109
  },
  "no_show": {
    "precision": 0.13233752620545072,
    "recall": 0.10540596952619495,
    "f1-score": 0.11734634599744395,
    "support": 4791
  },
  "yes": {
    "precision": 0.7240109427609428,
    "recall": 0.5604561189167175,
    "f1-score": 0.6318205816862934,
    "support": 24555
  },
  "accuracy": 0.5391711000514845,
  "macro avg": {
    "precision": 0.34669095593451316,
    "recall": 0.3449042873293088,
    "f1-score": 0.3389557086985473,
    "support": 46616
  },
  "weighted avg": {
    "precision": 0.5723707373673734,
    "recall": 0.5391711000514845,
    "f1-score": 0.5462973935282834,
    "support": 46616
  }
}
```